<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/StableDiffusionChapter20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
%pip install gradio
%pip install -U click
%pip install -U uvicorn

In [ ]:
import gradio

def greet(name):
    return "Hello " + name + "!"

demo = gradio.Interface(
    fn = greet
    , inputs = "text"
    , outputs = "text"
)

demo.launch()

In [ ]:
import gradio

def greet(name):
    return "Hello " + name + "!"

demo = gradio.Interface(
    fn = greet
    , inputs = "text"
    , outputs = "text"
)

gradio.close_all()

demo.launch(
    server_port = 7860
)

In [ ]:
import gradio
gradio.close_all()

def greet(name):
    return f"hello {name} !"

with gradio.Blocks() as demo:
    name_input      = gradio.Textbox(label = "Name")
    output          = gradio.Textbox(label = "output box")
    diffusion_btn   = gradio.Button("Generate")
    diffusion_btn.click(
        fn          = greet
        , inputs    = name_input
        , outputs   = output
    )

demo.launch(server_port = 7860)

In [ ]:
import gradio
gradio.close_all()

def greet(name, age):
    return f"hello {name} !", f"You age is {age}"

with gradio.Blocks() as demo:
    name_input    = gradio.Textbox(label = "Name")
    age_input     = gradio.Slider(minimum =0,maximum =100, label ="age slider")
    name_output   = gradio.Textbox(label = "name output box")
    age_output    = gradio.Textbox(label = "age output")
    diffusion_btn = gradio.Button("Generate")
    diffusion_btn.click(
        fn          = greet
        , inputs    = [name_input, age_input]
        , outputs   = [name_output, age_output]
    )

demo.launch()

In [ ]:
import gradio, time
gradio.close_all()

def my_function(text, progress=gradio.Progress()):
    for i in range(10):
        time.sleep(1)
        progress(i/10, desc=f"{i}")
    return text

with gradio.Blocks() as demo:
    input = gradio.Textbox()
    output = gradio.Textbox()
    btn = gradio.Button()
    btn.click(
        fn = my_function
        , inputs = input
        , outputs = output
    )

demo.queue().launch()

In [ ]:
!pip install diffusers
!pip install transformers scipy ftfy accelerate ipywidgets

In [ ]:
import gradio
gradio.close_all(verbose = True)

import torch
from diffusers import StableDiffusionPipeline

text2img_pipe = StableDiffusionPipeline.from_pretrained(
    "stablediffusionapi/deliberate-v2"
    , torch_dtype = torch.float16
    , safety_checker = None
).to("cuda:0")

def text2img(
    prompt:str
    , neg_prompt:str
    , progress_bar = gradio.Progress()
):
    return text2img_pipe(
        prompt = prompt
        , negative_prompt = neg_prompt
        , callback = (
            lambda step, timestep, latents:
                progress_bar(step/50, desc="denoising")
        )
    ).images[0]

with gradio.Blocks(
    theme = gradio.themes.Monochrome()
) as sd_app:
    gradio.Markdown("# Stable Diffusion in Gradio")
    prompt          = gradio.Textbox(label="Prompt", lines = 4)
    neg_prompt      = gradio.Textbox(label="Negative Prompt", lines = 2)
    sd_gen_btn      = gradio.Button("Generate Image")
    output_image    = gradio.Image()

    sd_gen_btn.click(
        fn = text2img
        , inputs = [prompt, neg_prompt]
        , outputs = output_image
    )

sd_app.queue().launch()